In [1]:
import numpy as np
import pandas as pd
import random
import os

## Read dataset

In [3]:
# Get the name of all the files in the output folder
folder_path = "/raid/sepideh/Project_MCL/1D-AEMpy-UW-metabolism-BM/output"
exc_files = ['obs' , # not added to final version - create different file for obs, for finetuning
             'meteorology_input.csv', # added to final version at the latest stage
             'area_input.csv', "depth_input.csv", "volume_input.csv", #include volume, area, and tp (duplicate for each depth), ignored the depth_input
             "tp_input.csv" , "tp_initial.csv" , # Added.
             'ice_final06', 'ko2_final06', 'secchi_final06'] # Duplicate for each depth
files = [f for f in os.listdir(folder_path) if all(excluded not in f for excluded in exc_files)]
len(files)

39

In [4]:
# Read the files and store them in a dictionary.
# Access the datasets: df["file name"]
df={}
df_melt = {}
for f in files:
    df[f] = pd.read_csv(os.path.join(folder_path, f))
    df_melt[f] = pd.melt(df[f], id_vars=["datetime"], var_name="depth", value_name=os.path.splitext(f)[0])


In [6]:
df_melt.keys()

dict_keys(['do_ax01.csv', 'do_bc02.csv', 'do_conv05.csv', 'do_diff04.csv', 'do_final06.csv', 'do_initial00.csv', 'do_pd03.csv', 'doc_final06.csv', 'docl_bc02.csv', 'docl_conv05.csv', 'docl_diff04.csv', 'docl_initial00.csv', 'docl_pd03.csv', 'docl_resp_pd03.csv', 'docr_bc02.csv', 'docr_conv05.csv', 'docr_diff04.csv', 'docr_initial00.csv', 'docr_pd03.csv', 'docr_resp_pd03.csv', 'kz_initial00.csv', 'npp_bc02.csv', 'poc_final06.csv', 'poc_resp_pd03.csv', 'pocl_bc02.csv', 'pocl_conv05.csv', 'pocl_diff04.csv', 'pocl_initial00.csv', 'pocl_pd03.csv', 'pocr_bc02.csv', 'pocr_conv05.csv', 'pocr_diff04.csv', 'pocr_initial00.csv', 'pocr_pd03.csv', 'temp_conv05.csv', 'temp_diff04.csv', 'temp_final06.csv', 'temp_initial00.csv', 'scaled_chla_final06.csv'])

In [8]:
df_melt['do_ax01.csv'].shape

(2190000, 3)

# Depth value adjustment

Some of the depth values are in scale of 0.5 but some of of them are in scale of 1. Here, we make them consistant in the dataset.

In [10]:

# Create a dictionary to map the values of depths in different formats
valid_depths = list(np.array(range(0, 50)).astype(float))
real_depths = list(np.arange(0, 25, 0.5))
depth_dict = dict(zip(real_depths, valid_depths))

# Iterate through the dictionary of DataFrames to update the depth values
for key, df in df_melt.items():
    df["depth"] = df["depth"].astype(float)

    if any((value) % 1 != 0 for value in df["depth"].unique()):
        df["depth"] = df["depth"].map(depth_dict)
print(key)
print(df['depth'].unique())


scaled_chla_final06.csv
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35.
 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49.]


## Merge datasets

In [11]:
merged_df = None
for key, df in df_melt.items():
    if merged_df is None:
        # Start with the first DataFrame
        merged_df = df
    else:
        # Merge with the existing DataFrame based on 'datetime' and 'depth'
        merged_df = pd.merge(merged_df, df, on=['datetime', 'depth'], how='outer')


In [12]:
# Check if columns have NaN values
nan_columns = merged_df.isna().any()

# Get the columns that have NaN values
columns_with_nan = nan_columns[nan_columns == True].index

# Display the columns with NaN values
print(columns_with_nan)

Index([], dtype='object')


In [13]:
print(merged_df.shape)
print(sorted(merged_df.columns))
merged_df

(2190000, 41)
['datetime', 'depth', 'do_ax01', 'do_bc02', 'do_conv05', 'do_diff04', 'do_final06', 'do_initial00', 'do_pd03', 'doc_final06', 'docl_bc02', 'docl_conv05', 'docl_diff04', 'docl_initial00', 'docl_pd03', 'docl_resp_pd03', 'docr_bc02', 'docr_conv05', 'docr_diff04', 'docr_initial00', 'docr_pd03', 'docr_resp_pd03', 'kz_initial00', 'npp_bc02', 'poc_final06', 'poc_resp_pd03', 'pocl_bc02', 'pocl_conv05', 'pocl_diff04', 'pocl_initial00', 'pocl_pd03', 'pocr_bc02', 'pocr_conv05', 'pocr_diff04', 'pocr_initial00', 'pocr_pd03', 'scaled_chla_final06', 'temp_conv05', 'temp_diff04', 'temp_final06', 'temp_initial00']


,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,pocr_bc02,pocr_conv05,pocr_diff04,pocr_initial00,pocr_pd03,temp_conv05,temp_diff04,temp_final06,temp_initial00,scaled_chla_final06
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,9.963798e+06,9.813901e+06,9.342519e+06,9.962500e+06,9.921259e+06,15.027714,15.643168,15.027714,15.800000,-0.101339
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,9.482548e+06,9.339806e+06,9.471131e+06,9.481250e+06,9.443247e+06,15.027714,15.236562,15.027714,15.600000,-0.101339
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,9.001298e+06,8.865710e+06,8.992584e+06,9.000000e+06,8.964666e+06,15.027714,14.996973,15.027714,15.400000,-0.101339
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,8.751298e+06,8.619426e+06,8.730964e+06,8.750000e+06,8.716487e+06,15.027714,14.695718,15.027714,15.200000,-0.101339
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,8.501298e+06,8.482163e+06,8.482669e+06,8.500000e+06,8.468185e+06,14.707939,14.417591,14.707939,14.800000,-0.101220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,4.567812e+04,4.598371e+04,4.598371e+04,4.567812e+04,4.557371e+04,7.536697,7.536697,7.536697,7.536653,-0.095987
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,3.863856e+04,3.896005e+04,3.896005e+04,3.863856e+04,3.855036e+04,7.532382,7.532382,7.532382,7.532475,-0.090439
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,3.181220e+04,3.213734e+04,3.213734e+04,3.181220e+04,3.174008e+04,7.486052,7.486052,7.486052,7.486889,-0.078666
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,2.539525e+04,2.571574e+04,2.571574e+04,2.539525e+04,2.534255e+04,7.055464,7.055464,7.055464,7.060261,0.112488


# Add Meteorological Data

In [14]:
# Get the start and end date of the merged dataset 
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'])

start_time = merged_df['datetime'].min()
end_time = merged_df['datetime'].max()

print(f"start date= {start_time}, end date= {end_time}")

start date= 2016-05-17 18:00:00, end date= 2021-05-16 17:00:00


In [15]:
# Read the dataset
df_meteorology_input = pd.read_csv(os.path.join(folder_path, "meteorology_input.csv"))

# There is one extra column in the dataset, we drop that.
df_meteorology_input.drop(columns="date", inplace=True)

# Fix the format of the datetime
df_meteorology_input['datetime'] = pd.to_datetime(df_meteorology_input['datetime'])
df_meteorology_input.shape

(70704, 13)

In [16]:
# Filter the meteorological dataset based on the date range of merged dataset
df_mtr_filtered = df_meteorology_input[(df_meteorology_input['datetime'] >= start_time) & (df_meteorology_input['datetime'] <= end_time)]
df_mtr_filtered.reset_index(drop=True,inplace=True)
df_mtr_filtered.shape

(43800, 13)

In [17]:
valid_depths = list(np.array(range(0, 50)).astype(float))

# Create DataFrame for depth values repeated for each time step
df_depths = pd.DataFrame(data={'depth':valid_depths * len(df_mtr_filtered)})

# Duplicate the meterolgical data for each depth
# Repeat each row by the number of depths = 50
df_meterological_data = pd.DataFrame(np.repeat(df_mtr_filtered.values, len(valid_depths), axis=0), columns=df_mtr_filtered.columns)
df_meterological_data= pd.concat([df_depths, df_meterological_data], ignore_index=False, axis=1)
df_meterological_data = df_meterological_data[['datetime', 'depth'] + [col for col in df_mtr_filtered.columns if col != 'datetime']]

# Merge the meterological data with other data
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'])
df_meterological_data['datetime'] = pd.to_datetime(df_meterological_data['datetime'])

df_final = pd.merge(merged_df, df_meterological_data, on=['datetime', 'depth'], how='outer')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Longwave_Radiation_Downwelling_wattPerMeterSquared,Relative_Humidity_percent,Ten_Meter_Elevation_Wind_Speed_meterPerSecond,Precipitation_millimeterPerDay,Surface_Level_Barometric_Pressure_pascal,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,322.450012,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,322.450012,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,322.450012,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,322.450012,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,322.450012,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,342.149994,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,342.149994,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,342.149994,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,342.149994,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17


In [14]:
df_final.columns

Index(['datetime', 'depth', 'do_ax01', 'do_bc02', 'do_conv05', 'do_diff04',
       'do_final06', 'do_initial00', 'do_pd03', 'doc_final06', 'docl_bc02',
       'docl_conv05', 'docl_diff04', 'docl_initial00', 'docl_pd03',
       'docl_resp_pd03', 'docr_bc02', 'docr_conv05', 'docr_diff04',
       'docr_initial00', 'docr_pd03', 'docr_resp_pd03', 'kz_initial00',
       'npp_bc02', 'poc_final06', 'poc_resp_pd03', 'pocl_bc02', 'pocl_conv05',
       'pocl_diff04', 'pocl_initial00', 'pocl_pd03', 'pocr_bc02',
       'pocr_conv05', 'pocr_diff04', 'pocr_initial00', 'pocr_pd03',
       'temp_conv05', 'temp_diff04', 'temp_final06', 'temp_initial00',
       'Air_Temperature_celsius',
       'Shortwave_Radiation_Downwelling_wattPerMeterSquared',
       'Longwave_Radiation_Downwelling_wattPerMeterSquared',
       'Relative_Humidity_percent',
       'Ten_Meter_Elevation_Wind_Speed_meterPerSecond',
       'Precipitation_millimeterPerDay',
       'Surface_Level_Barometric_Pressure_pascal', 'Cloud_Cover', 

## Add volume data

In [19]:
# Read the dataset
df_volume = pd.read_csv(os.path.join(folder_path, "volume_input.csv"))
df_volume = df_volume.rename(columns={'0': 'volume_input'})

# Add depth column to the dataset
df_volume['depth'] = valid_depths

df_volume

,volume_input,depth
0,19925000.0,0.0
1,18962500.0,1.0
2,18000000.0,2.0
3,17500000.0,3.0
4,17000000.0,4.0
5,16525000.0,5.0
6,16050000.0,6.0
7,15600000.0,7.0
8,15150000.0,8.0
9,14750000.0,9.0


In [20]:


# Merge with final dataset
df_final = pd.merge(df_final, df_volume, on='depth', how='inner')
df_final


,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Relative_Humidity_percent,Ten_Meter_Elevation_Wind_Speed_meterPerSecond,Precipitation_millimeterPerDay,Surface_Level_Barometric_Pressure_pascal,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list,volume_input
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,19925000.0
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18962500.0
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18000000.0
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17500000.0
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,64.306063,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,1275000.0
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,800000.0
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,425000.0
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,63.149491,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,50000.0


## Add area data

In [21]:

# Read the dataset
df_area = pd.read_csv(os.path.join(folder_path, "area_input.csv"))
df_area = df_area.rename(columns={'0': 'area_input'})

# Add depth column to the dataset
df_area['depth'] = valid_depths
df_area.head()

,area_input,depth
0,38887500.0,0.0
1,36962500.0,1.0
2,35500000.0,2.0
3,34500000.0,3.0
4,33525000.0,4.0


In [22]:


# Merge with final dataset
df_final = pd.merge(df_final, df_area, on='depth', how='inner')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Ten_Meter_Elevation_Wind_Speed_meterPerSecond,Precipitation_millimeterPerDay,Surface_Level_Barometric_Pressure_pascal,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,19925000.0,38887500.0
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18962500.0,36962500.0
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18000000.0,35500000.0
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17500000.0,34500000.0
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,3.327762,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17000000.0,33525000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,1275000.0,2075000.0
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,800000.0,1225000.0
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,425000.0,475000.0
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,4.378333,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,50000.0,75000.0


## Add ice data

In [23]:

# Read the dataset
df_ice = pd.read_csv(os.path.join(folder_path, "ice_final06.csv"))
df_ice = df_ice.rename(columns={'0.0': 'ice_final06'})
df_ice['datetime'] = pd.to_datetime(df_ice['datetime'])


# Create DataFrame for depth values repeated for each time step
df_depths = pd.DataFrame(data={'depth':valid_depths * len(df_ice)})


# Duplicate ice data for each depth
df_ice = pd.DataFrame(np.repeat(df_ice.values, len(valid_depths), axis=0), columns=df_ice.columns)
df_ice= pd.concat([df_depths, df_ice], ignore_index=False, axis=1)
df_ice

,depth,datetime,ice_final06
0,0.0,2016-05-17 18:00:00,0.0
1,1.0,2016-05-17 18:00:00,0.0
2,2.0,2016-05-17 18:00:00,0.0
3,3.0,2016-05-17 18:00:00,0.0
4,4.0,2016-05-17 18:00:00,0.0
...,...,...,...
2189995,45.0,2021-05-16 17:00:00,0.0
2189996,46.0,2021-05-16 17:00:00,0.0
2189997,47.0,2021-05-16 17:00:00,0.0
2189998,48.0,2021-05-16 17:00:00,0.0


In [24]:

df_final = pd.merge(df_final, df_ice, on=['datetime', 'depth'], how='outer')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Precipitation_millimeterPerDay,Surface_Level_Barometric_Pressure_pascal,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input,ice_final06
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,19925000.0,38887500.0,0.0
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18962500.0,36962500.0,0.0
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,18000000.0,35500000.0,0.0
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17500000.0,34500000.0,0.0
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,0.0,98914.82031,1.0,11923201.0,10.359133,138,18,17000000.0,33525000.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,1275000.0,2075000.0,0.0
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,800000.0,1225000.0,0.0
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,425000.0,475000.0,0.0
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,0.0,98641.21875,0.755736,169599601.0,15.977638,136,17,50000.0,75000.0,0.0


## Add ko2 data

In [25]:

# Read the dataset
df_ko2 = pd.read_csv(os.path.join(folder_path, "ko2_final06.csv"))
df_ko2 = df_ko2.rename(columns={'0.0': 'ko2_final06'})
df_ko2['datetime'] = pd.to_datetime(df_ko2['datetime'])

# There is one nan value, set it to zero
df_ko2['ko2_final06'] = df_ko2['ko2_final06'].fillna(0)

# Create DataFrame for depth values repeated for each time step
df_depths = pd.DataFrame(data={'depth':valid_depths * len(df_ko2)})


# Duplicate ko2 data for each depth
df_ko2 = pd.DataFrame(np.repeat(df_ko2.values, len(valid_depths), axis=0), columns=df_ko2.columns)
df_ko2= pd.concat([df_depths, df_ko2], ignore_index=False, axis=1)
df_ko2

,depth,datetime,ko2_final06
0,0.0,2016-05-17 18:00:00,0.0
1,1.0,2016-05-17 18:00:00,0.0
2,2.0,2016-05-17 18:00:00,0.0
3,3.0,2016-05-17 18:00:00,0.0
4,4.0,2016-05-17 18:00:00,0.0
...,...,...,...
2189995,45.0,2021-05-16 17:00:00,0.043174
2189996,46.0,2021-05-16 17:00:00,0.043174
2189997,47.0,2021-05-16 17:00:00,0.043174
2189998,48.0,2021-05-16 17:00:00,0.043174


In [26]:
# Merge with final dataset
df_final = pd.merge(df_final, df_ko2, on=['datetime', 'depth'], how='outer')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Surface_Level_Barometric_Pressure_pascal,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input,ice_final06,ko2_final06
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,98914.82031,1.0,11923201.0,10.359133,138,18,19925000.0,38887500.0,0.0,0.0
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,98914.82031,1.0,11923201.0,10.359133,138,18,18962500.0,36962500.0,0.0,0.0
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,98914.82031,1.0,11923201.0,10.359133,138,18,18000000.0,35500000.0,0.0,0.0
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,98914.82031,1.0,11923201.0,10.359133,138,18,17500000.0,34500000.0,0.0,0.0
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,98914.82031,1.0,11923201.0,10.359133,138,18,17000000.0,33525000.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,98641.21875,0.755736,169599601.0,15.977638,136,17,1275000.0,2075000.0,0.0,0.043174
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,98641.21875,0.755736,169599601.0,15.977638,136,17,800000.0,1225000.0,0.0,0.043174
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,98641.21875,0.755736,169599601.0,15.977638,136,17,425000.0,475000.0,0.0,0.043174
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,98641.21875,0.755736,169599601.0,15.977638,136,17,50000.0,75000.0,0.0,0.043174


## Add secchi data

In [27]:
pd.read_csv(os.path.join(folder_path, "secchi_final06.csv"))

,datetime,0.0
0,2016-05-17 18:00:00,1.805759
1,2016-05-17 19:00:00,1.809723
2,2016-05-17 20:00:00,1.815501
3,2016-05-17 21:00:00,1.821597
4,2016-05-17 22:00:00,1.827775
...,...,...
43795,2021-05-16 13:00:00,2.492376
43796,2021-05-16 14:00:00,2.481516
43797,2021-05-16 15:00:00,2.468963
43798,2021-05-16 16:00:00,2.460331


In [28]:

# Read the dataset
df_secchi = pd.read_csv(os.path.join(folder_path, "secchi_final06.csv"))
df_secchi = df_secchi.rename(columns={'0.0': 'secchi_final06'})
df_secchi['datetime'] = pd.to_datetime(df_secchi['datetime'])


# Create DataFrame for depth values repeated for each time step
df_depths = pd.DataFrame(data={'depth':valid_depths * len(df_secchi)})


# Duplicate ice data for each depth
df_secchi = pd.DataFrame(np.repeat(df_secchi.values, len(valid_depths), axis=0), columns=df_secchi.columns)
df_secchi= pd.concat([df_depths, df_secchi], ignore_index=False, axis=1)
df_secchi

,depth,datetime,secchi_final06
0,0.0,2016-05-17 18:00:00,1.805759
1,1.0,2016-05-17 18:00:00,1.805759
2,2.0,2016-05-17 18:00:00,1.805759
3,3.0,2016-05-17 18:00:00,1.805759
4,4.0,2016-05-17 18:00:00,1.805759
...,...,...,...
2189995,45.0,2021-05-16 17:00:00,2.453275
2189996,46.0,2021-05-16 17:00:00,2.453275
2189997,47.0,2021-05-16 17:00:00,2.453275
2189998,48.0,2021-05-16 17:00:00,2.453275


In [29]:
# Merge with final dataset
df_final = pd.merge(df_final, df_secchi, on=['datetime', 'depth'], how='outer')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,Cloud_Cover,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input,ice_final06,ko2_final06,secchi_final06
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,1.0,11923201.0,10.359133,138,18,19925000.0,38887500.0,0.0,0.0,1.805759
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,1.0,11923201.0,10.359133,138,18,18962500.0,36962500.0,0.0,0.0,1.805759
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,1.0,11923201.0,10.359133,138,18,18000000.0,35500000.0,0.0,0.0,1.805759
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,1.0,11923201.0,10.359133,138,18,17500000.0,34500000.0,0.0,0.0,1.805759
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,1.0,11923201.0,10.359133,138,18,17000000.0,33525000.0,0.0,0.0,1.805759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,0.755736,169599601.0,15.977638,136,17,1275000.0,2075000.0,0.0,0.043174,2.453275
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,0.755736,169599601.0,15.977638,136,17,800000.0,1225000.0,0.0,0.043174,2.453275
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,0.755736,169599601.0,15.977638,136,17,425000.0,475000.0,0.0,0.043174,2.453275
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,0.755736,169599601.0,15.977638,136,17,50000.0,75000.0,0.0,0.043174,2.453275


## Add tp_initial data

In [30]:
pd.read_csv(os.path.join(folder_path, "tp_initial.csv"))

,0
0,37.097143
1,37.120952
2,37.144762
3,37.168571
4,37.192381
...,...
43795,73.256508
43796,73.288254
43797,73.320000
43798,73.351746


In [31]:
# Read the dataset
df_tp = pd.read_csv(os.path.join(folder_path, "tp_initial.csv"))
df_tp = df_tp.rename(columns={'0': 'tp_initial'})


# Create DataFrame for depth values repeated for each time step
df_depths = pd.DataFrame(data={'depth':valid_depths * len(df_tp)})

# # Duplicate tp data for each depth
df_tp = pd.DataFrame(np.repeat(df_tp.values, len(valid_depths), axis=0), columns=df_tp.columns)
df_tp['datetime'] = pd.to_datetime(df_final['datetime'])
df_tp= pd.concat([df_depths, df_tp], ignore_index=False, axis=1)
df_tp

,depth,tp_initial,datetime
0,0.0,37.097143,2016-05-17 18:00:00
1,1.0,37.097143,2016-05-17 18:00:00
2,2.0,37.097143,2016-05-17 18:00:00
3,3.0,37.097143,2016-05-17 18:00:00
4,4.0,37.097143,2016-05-17 18:00:00
...,...,...,...
2189995,45.0,73.383492,2021-05-16 17:00:00
2189996,46.0,73.383492,2021-05-16 17:00:00
2189997,47.0,73.383492,2021-05-16 17:00:00
2189998,48.0,73.383492,2021-05-16 17:00:00


In [32]:

df_final = pd.merge(df_final, df_tp, on=['datetime', 'depth'], how='outer')
df_final

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input,ice_final06,ko2_final06,secchi_final06,tp_initial
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,11923201.0,10.359133,138,18,19925000.0,38887500.0,0.0,0.0,1.805759,37.097143
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,11923201.0,10.359133,138,18,18962500.0,36962500.0,0.0,0.0,1.805759,37.097143
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,11923201.0,10.359133,138,18,18000000.0,35500000.0,0.0,0.0,1.805759,37.097143
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,11923201.0,10.359133,138,18,17500000.0,34500000.0,0.0,0.0,1.805759,37.097143
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,11923201.0,10.359133,138,18,17000000.0,33525000.0,0.0,0.0,1.805759,37.097143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,169599601.0,15.977638,136,17,1275000.0,2075000.0,0.0,0.043174,2.453275,73.383492
2189996,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,169599601.0,15.977638,136,17,800000.0,1225000.0,0.0,0.043174,2.453275,73.383492
2189997,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,169599601.0,15.977638,136,17,425000.0,475000.0,0.0,0.043174,2.453275,73.383492
2189998,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,169599601.0,15.977638,136,17,50000.0,75000.0,0.0,0.043174,2.453275,73.383492


## Save

In [33]:
# Check if there is any nan values
df_final.isna().sum().sum()

0

In [34]:
folder_path = '/raid/sepideh/Project_MCL/1D-AEMpy-UW-metabolism-BM'
output_path = os.path.join(folder_path, "all_data_lake_modeling_process_based.csv")
df_final.to_csv(output_path, index=False)

In [35]:
df_final.head()

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,dt,ea,day_of_year_list,time_of_day_list,volume_input,area_input,ice_final06,ko2_final06,secchi_final06,tp_initial
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,231130000.0,2.227493e+08,5.943507,...,11923201.0,10.359133,138,18,19925000.0,38887500.0,0.0,0.0,1.805759,37.097143
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,219965000.0,2.201040e+08,5.943492,...,11923201.0,10.359133,138,18,18962500.0,36962500.0,0.0,0.0,1.805759,37.097143
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,209700000.0,2.096759e+08,5.943475,...,11923201.0,10.359133,138,18,18000000.0,35500000.0,0.0,0.0,1.805759,37.097143
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,205625000.0,2.055109e+08,5.943465,...,11923201.0,10.359133,138,18,17500000.0,34500000.0,0.0,0.0,1.805759,37.097143
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,201450000.0,2.012874e+08,5.854266,...,11923201.0,10.359133,138,18,17000000.0,33525000.0,0.0,0.0,1.805759,37.097143


In [32]:
# df_final[df_final["datetime"]=="2021-05-15 20:00:00"]["secchi_final06"]